In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# PhishGuard AI - Model Training (Google Colab)\\n",
    "This notebook trains a phishing URL detector and exports `model.pkl`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install -q pandas scikit-learn xgboost joblib"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import ipaddress, re, joblib\\n",
    "import pandas as pd\\n",
    "from urllib.parse import urlparse\\n",
    "from sklearn.model_selection import train_test_split\\n",
    "from sklearn.metrics import classification_report, accuracy_score\\n",
    "from sklearn.ensemble import RandomForestClassifier\\n",
    "\\n",
    "LABEL_MAP = {\\n",
    "    'phishing': 1, 'bad': 1, 'malicious': 1, '1': 1, '-1': 1,\\n",
    "    'benign': 0, 'good': 0, 'legitimate': 0, '0': 0\\n",
    "}\\n",
    "\\n",
    "IPV4_PATTERN = re.compile(r'^(\\\\d{1,3}\\\\.){3}\\\\d{1,3}$')\\n",
    "\\n",
    "def normalize_labels(series):\\n",
    "    norm = series.astype(str).str.strip().str.lower()\\n",
    "    mapped = norm.map(LABEL_MAP)\\n",
    "    if mapped.isna().any():\\n",
    "        raise ValueError(f'Unsupported labels: {sorted(norm[mapped.isna()].unique())[:10]}')\\n",
    "    return mapped.astype('int8')\\n",
    "\\n",
    "def has_ip(netloc):\\n",
    "    host = netloc.split(':')[0].strip('[]')\\n",
    "    if not host:\\n",
    "        return 0\\n",
    "    if IPV4_PATTERN.match(host):\\n",
    "        try:\\n",
    "            ipaddress.ip_address(host)\\n",
    "            return 1\\n",
    "        except ValueError:\\n",
    "            return 0\\n",
    "    try:\\n",
    "        ipaddress.ip_address(host)\\n",
    "        return 1\\n",
    "    except ValueError:\\n",
    "        return 0\\n",
    "\\n",
    "def extract_features(urls):\\n",
    "    urls = urls.fillna('').astype(str).str.strip()\\n",
    "    prep = urls.where(urls.str.contains('://'), 'http://' + urls)\\n",
    "    parsed = prep.map(urlparse)\\n",
    "    return pd.DataFrame({\\n",
    "        'url_length': prep.str.len(),\\n",
    "        'num_dots': prep.str.count(r'\\\\.'),\\n",
    "        'num_hyphens': prep.str.count('-'),\\n",
    "        'num_digits': prep.str.count(r'\\\\d'),\\n",
    "        'num_slashes': prep.str.count('/'),\\n",
    "        'has_at_symbol': prep.str.contains('@', regex=False).astype('int8'),\\n",
    "        'uses_https': parsed.map(lambda p: p.scheme.lower() == 'https').astype('int8'),\\n",
    "        'has_ip_in_domain': parsed.map(lambda p: has_ip(p.netloc)).astype('int8')\\n",
    "    })\\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Upload dataset with columns: url + label\\n",
    "from google.colab import files\\n",
    "uploaded = files.upload()\\n",
    "dataset_path = next(iter(uploaded.keys()))\\n",
    "df = pd.read_csv(dataset_path, low_memory=False)\\n",
    "\\n",
    "url_col = 'url'\\n",
    "label_col = [c for c in df.columns if c.lower() in {'label','labels','class','target','status'}][0]\\n",
    "\\n",
    "X = extract_features(df[url_col])\\n",
    "y = normalize_labels(df[label_col])\\n",
    "\\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\\n",
    "\\n",
    "model = RandomForestClassifier(\\n",
    "    n_estimators=500,\\n",
    "    max_depth=30,\\n",
    "    min_samples_split=5,\\n",
    "    class_weight='balanced_subsample',\\n",
    "    n_jobs=-1,\\n",
    "    random_state=42\\n",
    ")\\n",
    "model.fit(X_train, y_train)\\n",
    "pred = model.predict(X_test)\\n",
    "print('Accuracy:', accuracy_score(y_test, pred))\\n",
    "print(classification_report(y_test, pred))\\n",
    "joblib.dump(model, 'model.pkl')\\n",
    "files.download('model.pkl')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}